# Texas RRC Form R-1 Extraction Pipeline

**Extraction tiers:** Tier 1 (pdfplumber, digital) → Tier 2 (Tesseract OCR, scans) → Tier 3 (Claude API vision + retry)

**Crash-safe:** Results saved after every PDF. Restart picks up where you left off. No duplicate API calls.

### Setup
```bash
pip install pdfplumber pytesseract Pillow numpy scipy pandas requests beautifulsoup4 openpyxl PyMuPDF anthropic
```
Windows: install Tesseract from https://github.com/UB-Mannheim/tesseract/wiki

```bash
set ANTHROPIC_API_KEY=sk-ant-...
```

In [1]:
import os, re, json, base64, io, tempfile, subprocess, time, traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
from PIL import Image
from scipy.signal import find_peaks


from dotenv import load_dotenv

# Load credentials from XX/.env — one level up from this notebook's directory
load_dotenv(dotenv_path=Path('.env'))

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '').strip("'\"")

try:
    import fitz
    HAS_FITZ = True
except ImportError:
    HAS_FITZ = False
    print('WARNING: pip install PyMuPDF')

try:
    from anthropic import Anthropic
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False
    print('INFO: pip install anthropic (Tier 3 disabled)')

RAW_DATA_DIR = Path('Raw Data/Texas RRC')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

EXISTING_DB_PATH = 'output_cleaned_v2_1.xlsx'

# Windows Tesseract — uncomment and adjust:
pytesseract.pytesseract.tesseract_cmd = r'C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

PRODUCT_CODES = {1:'Propane',2:'Butane',3:'Butane-Propane',4:'Motor Gasoline',5:'Kerosene',6:'Home Heating Oil',7:'Diesel Fuel',8:'Other Middle Distillates',9:'Aviation Gasoline',10:'Kerosene-Type Jet Fuel',11:'Naphtha-Type Jet Fuel',12:'Fuel Oils #4 For Utility Use',13:'Fuel Oils #5,#6 For Utility Use',14:'Fuel Oils #4 For Non-Utility Use',15:'Fuel Oils #5,#6 For Non-Utility Use',16:'Bunker C',17:'Navy Special',18:'Other Residual Fuels',19:'Petrochemical Feedstock',20:'Lubricants',21:'Special Naphthas',22:'Solvent Products',23:'Miscellaneous',24:'Crude Oil'}
DATA_COLUMNS = ['Storage Beginning of Month','Receipts','Input','Products Manufactured','Fuel Used','Deliveries','Storage End of Month']

print(f'Ready. PyMuPDF={HAS_FITZ}, Claude={HAS_ANTHROPIC}')

Ready. PyMuPDF=True, Claude=True


---
## 1. Extraction Engine

In [3]:
# ── Helpers ──

def parse_number(raw) -> float:
    if raw is None: return 0.0
    s = str(raw).strip().replace(',','').replace(' ','').replace('|','')
    s = re.sub(r'[^0-9.\-]', '', s)
    if not s or s == '-': return 0.0
    try: return float(s)
    except: return 0.0

def rasterize_page(pdf_path, dpi=300):
    if HAS_FITZ:
        doc = fitz.open(pdf_path)
        zoom = dpi / 72
        pix = doc[0].get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        doc.close()
        return img
    with tempfile.TemporaryDirectory() as d:
        subprocess.run(['pdftoppm','-jpeg','-r',str(dpi),'-f','1','-l','1',pdf_path,f'{d}/p'],check=True,capture_output=True)
        return Image.open(sorted(Path(d).glob('p-*.jpg'))[0]).copy()

def image_to_base64(img, max_bytes=4_500_000):
    """Compress image to JPEG base64 under API size limit."""
    for quality in [90, 75, 60, 45]:
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        if buf.tell() <= max_bytes:
            return base64.b64encode(buf.getvalue()).decode('utf-8')
    scale = 0.75
    while scale > 0.3:
        resized = img.resize((int(img.width*scale), int(img.height*scale)), Image.LANCZOS)
        buf = io.BytesIO()
        resized.save(buf, format='JPEG', quality=60)
        if buf.tell() <= max_bytes:
            return base64.b64encode(buf.getvalue()).decode('utf-8')
        scale -= 0.1
    buf = io.BytesIO()
    img.resize((int(img.width*0.3), int(img.height*0.3)), Image.LANCZOS).save(buf, format='JPEG', quality=50)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

def is_digital_pdf(path):
    try:
        with pdfplumber.open(path) as pdf:
            return len((pdf.pages[0].extract_text() or '').strip()) > 200
    except: return False

def extract_header_text(text):
    h = {}
    m = re.search(r'Operator[:\s]+(.+?)(?:Month|Address|$)', text, re.IGNORECASE)
    if m: h['operator'] = m.group(1).strip().rstrip('.')
    m = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}', text, re.IGNORECASE)
    if m: h['month_str'] = m.group(0)
    m = re.search(r'(\d{2}-\d{4})', text)
    if m: h['registration_no'] = m.group(1)
    return h

def validate_balance(df):
    df = df.copy()
    df['balance_diff'] = (df['Storage Beginning of Month']+df['Receipts']+df['Products Manufactured']) - (df['Input']+df['Fuel Used']+df['Deliveries']+df['Storage End of Month'])
    df['balanced'] = df['balance_diff'].abs() < 2
    return df

def balance_stats(df):
    active = df[df[DATA_COLUMNS].sum(axis=1)>0]
    n = len(active)
    b = int(active['balanced'].sum()) if n else 0
    return {'total_active':n,'balanced_count':b,'pass_rate':round(b/n,3) if n else 1.0}

print('Helpers loaded.')

Helpers loaded.


In [4]:
# ── Tier 1: Digital PDF ──

def extract_digital(path):
    with pdfplumber.open(path) as pdf:
        text = pdf.pages[0].extract_text() or ''
        header = extract_header_text(text)
        tables = pdf.pages[0].extract_tables()
        if not tables:
            return {'header':header,'data':[],'mode':'digital','tier':3,'error':'No table','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
        rows = []
        for r in tables[0][1:]:
            if r[0] and 'TOTALS' in str(r[0]): continue
            c = parse_number(r[1])
            if c == 0: continue
            rows.append({'Code':int(c),'Storage Beginning of Month':parse_number(r[2]),'Receipts':parse_number(r[3]),'Input':parse_number(r[4]),'Products Manufactured':parse_number(r[5]),'Fuel Used':parse_number(r[6]),'Deliveries':parse_number(r[7]),'Storage End of Month':parse_number(r[8])})
        df = validate_balance(pd.DataFrame(rows)) if rows else pd.DataFrame()
        return {'header':header,'data':rows,'mode':'digital','tier':1,'error':None,
                'balance_stats':balance_stats(df) if not df.empty else {},'api_usage':{'cost_usd':0,'calls':0}}

print('Tier 1 loaded.')

Tier 1 loaded.


In [5]:
# ── Tier 2: Tesseract OCR ──

def detect_grid(img):
    arr = np.array(img.convert('L')); th = 100
    rk = (arr<th).sum(axis=1)

    # Try increasingly relaxed thresholds before giving up.
    # Tight thresholds work for clean scans; loose ones catch degraded ones.
    for h_frac, min_dist, min_hp, min_di, v_frac in [
        (0.15, 12, 10, 5, 0.25),   # original (strict)
        (0.10,  8,  8, 4, 0.18),   # moderate
        (0.07,  6,  6, 3, 0.12),   # loose
    ]:
        hp, _ = find_peaks(rk, height=arr.shape[1]*h_frac, distance=min_dist)
        hp = hp[(hp >= arr.shape[0]*0.15) & (hp <= arr.shape[0]*0.9)]
        if len(hp) < min_hp:
            continue
        sp = np.diff(hp); di = np.where((sp >= 18) & (sp <= 50))[0]
        if len(di) < min_di:
            continue
        thl = hp[max(0, di[0]-2):min(len(hp), di[-1]+3)].tolist()
        tr = arr[thl[0]:thl[-1], :]; ck = (tr < th).sum(axis=0)
        vp, _ = find_peaks(ck, height=len(tr)*v_frac, distance=25)
        return vp.tolist(), thl

    return None, None

def id_columns(vl):
    if len(vl)<4: return []
    gaps=[(vl[i],vl[i+1],vl[i+1]-vl[i]) for i in range(len(vl)-1)]
    wide=[(a,b,w) for a,b,w in gaps if w>100]
    if len(wide)>=8: return [(a,b) for a,b,w in wide[1:8]]
    if len(wide)==7: return [(a,b) for a,b,w in wide]
    return [(a,b) for a,b,w in gaps[-7:]] if len(gaps)>=7 else []

def find_start(hl,vl,img):
    if not vl or len(vl)<4: return min(2,len(hl)-25)
    gaps=[(vl[i],vl[i+1],vl[i+1]-vl[i],i) for i in range(len(vl)-1)]
    lh=[g for g in gaps if g[0]<img.width*0.4]
    if not lh: return min(2,len(hl)-25)
    mi=max(lh,key=lambda g:g[2])[3]
    cx1,cx2=vl[mi+1],vl[min(mi+2,len(vl)-1)]
    for i in range(min(15,len(hl)-1)):
        y1,y2=hl[i]+2,hl[i+1]-2
        if y2-y1<8: continue
        c=img.crop((cx1,y1,cx2,y2)); w,h=c.size
        if w<5 or h<5: continue
        c=c.resize((w*4,h*4),Image.LANCZOS)
        t=pytesseract.image_to_string(c,config='--psm 7 -c tessedit_char_whitelist=0123456789').strip()
        if t=='1': return i
    sp=[hl[i+1]-hl[i] for i in range(len(hl)-1)]
    for i in range(1,len(sp)):
        if sp[i]<45 and sp[i-1]>=45: return i
    return min(2,max(0,len(hl)-26))

def extract_scanned(path, dpi=200):
    try: img=rasterize_page(path,dpi)
    except Exception as e:
        return {'header':{},'data':[],'mode':'scanned_ocr','tier':3,'error':str(e),'balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    header=extract_header_text(pytesseract.image_to_string(img.crop((0,0,img.width,int(img.height*0.25)))))
    vl,hl=detect_grid(img)
    if not vl or not hl:
        return {'header':header,'data':[],'mode':'scanned_ocr','tier':3,'error':'No grid','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    cb=id_columns(vl)
    if len(cb)<7:
        return {'header':header,'data':[],'mode':'scanned_ocr','tier':3,'error':f'{len(cb)} cols','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    ds=find_start(hl,vl,img); PAD,SCALE=3,4; rows=[]
    for code in range(1,25):
        hi=ds+code-1
        if hi+1>=len(hl): break
        yt,yb=hl[hi]+PAD,hl[hi+1]-PAD
        if yb<=yt: continue
        rd={'Code':code}
        for ci,cn in enumerate(DATA_COLUMNS):
            if ci>=len(cb): rd[cn]=0.0; continue
            x1,x2=cb[ci]; cell=img.crop((x1+PAD,yt,x2-PAD,yb)); w,h=cell.size
            if w<5 or h<5: rd[cn]=0.0; continue
            cell=cell.resize((w*SCALE,h*SCALE),Image.LANCZOS)
            t=pytesseract.image_to_string(cell,config='--psm 7 -c tessedit_char_whitelist=0123456789,.-').strip()
            rd[cn]=parse_number(t)
        rows.append(rd)
    existing={r['Code'] for r in rows}
    for c in range(1,25):
        if c not in existing: rows.append({**{col:0.0 for col in DATA_COLUMNS},'Code':c})
    rows.sort(key=lambda r:r['Code'])
    df=validate_balance(pd.DataFrame(rows)); st=balance_stats(df)
    tier=1 if st['pass_rate']>=0.75 else 2 if st['pass_rate']>=0.25 else 3
    return {'header':header,'data':rows,'mode':'scanned_ocr','tier':tier,'error':None,'balance_stats':st,'api_usage':{'cost_usd':0,'calls':0}}

print('Tier 2 loaded.')

Tier 2 loaded.


In [6]:
# ── Tier 3: Claude API with retry ──

CLAUDE_PROMPT = '''Extract the material balance table from this Texas RRC Form R-1.

The table has EXACTLY these 24 rows in this order:
1=Propane, 2=Butane, 3=Butane-Propane, 4=Motor Gasoline, 5=Kerosene,
6=Home Heating Oil, 7=Diesel Fuel, 8=Other Middle Distillates, 9=Aviation Gasoline,
10=Kerosene-Type Jet Fuel, 11=Naphtha-Type Jet Fuel, 12=Fuel Oils #4 Utility,
13=Fuel Oils #5,#6 Utility, 14=Fuel Oils #4 Non-Utility, 15=Fuel Oils #5,#6 Non-Utility,
16=Bunker C, 17=Navy Special, 18=Other Residual Fuels, 19=Petrochemical Feedstock,
20=Lubricants, 21=Special Naphthas, 22=Solvent Products, 23=Miscellaneous, 24=Crude Oil

Columns: Storage Beginning of Month, Receipts, Input Runs to Stills and/or Blends,
Products Manufactured, Fuel Used, Deliveries, Storage End of Month.

Return ONLY JSON — no markdown, no backticks:
{"operator":"...","month":"...","registration_no":"XX-XXXX","refinery_name":"...",
"rows":[{"code":1,"storage_bom":0,"receipts":0,"input":0,"products_mfg":0,"fuel_used":0,"deliveries":0,"storage_eom":0}]}

Rules:
- Use 0 for blank/dash cells. Integers only, no commas.
- EVERY row must match the product name printed on that line.
- Balance: storage_bom + receipts + products_mfg = input + fuel_used + deliveries + storage_eom
- If a number seems wrong, re-read that row carefully before committing.'''

# Errors worth retrying (transient); all others bubble up immediately.
_RETRYABLE = ('RateLimitError', 'APITimeoutError', 'APIConnectionError', 'InternalServerError')

def _call_claude(b64, prompt, model, max_retries=3):
    """Single Claude API call with exponential backoff for transient errors.
    Returns (message, error_string).
    """
    last_err = None
    for attempt in range(max_retries):
        try:
            msg = Anthropic().messages.create(model=model, max_tokens=4096, messages=[{
                'role': 'user', 'content': [
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b64}},
                    {'type': 'text', 'text': prompt}]}])
            return msg, None
        except Exception as e:
            last_err = e
            err_type = type(e).__name__
            is_retryable = any(r in err_type for r in _RETRYABLE)
            if is_retryable and attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)  # 2s, 4s, 8s
                print(f'    Claude {err_type} (attempt {attempt+1}/{max_retries}), retrying in {wait}s...')
                time.sleep(wait)
            else:
                break
    return None, f'{type(last_err).__name__}: {last_err}'


def _parse_claude_response(msg):
    """Parse Claude message into header + rows. Returns (header, rows, error)."""
    txt = re.sub(r'^```(?:json)?\s*','', msg.content[0].text.strip())
    txt = re.sub(r'\s*```$','', txt)
    try:
        p = json.loads(txt)
    except json.JSONDecodeError as e:
        return {}, [], f'JSON parse: {e}'
    header = {'operator':p.get('operator',''),'month_str':p.get('month',''),
              'registration_no':p.get('registration_no',''),'refinery_name':p.get('refinery_name','')}
    rows = []
    for r in p.get('rows',[]):
        c = int(r.get('code',0))
        if c<1 or c>24: continue
        rows.append({'Code':c,'Storage Beginning of Month':float(r.get('storage_bom',0)),
            'Receipts':float(r.get('receipts',0)),'Input':float(r.get('input',0)),
            'Products Manufactured':float(r.get('products_mfg',0)),'Fuel Used':float(r.get('fuel_used',0)),
            'Deliveries':float(r.get('deliveries',0)),'Storage End of Month':float(r.get('storage_eom',0))})
    # Fill missing codes
    existing = {r['Code'] for r in rows}
    for c in range(1,25):
        if c not in existing: rows.append({**{col:0.0 for col in DATA_COLUMNS},'Code':c})
    rows.sort(key=lambda r:r['Code'])
    return header, rows, None


def extract_claude_with_retry(path, model='claude-sonnet-4-6'):
    """Tier 3: Claude vision + targeted retry for unbalanced rows."""
    if not HAS_ANTHROPIC:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,
                'error':'anthropic not installed','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    
    # Rasterize
    try:
        img = rasterize_page(path, 300)
        b64 = image_to_base64(img)
    except Exception as e:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,
                'error':f'Image: {e}','balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    
    # ── Pass 1 ──
    msg1, err1 = _call_claude(b64, CLAUDE_PROMPT, model)
    if err1:
        return {'header':{},'data':[],'mode':'claude_vision','tier':3,
                'error':err1,'balance_stats':{},'api_usage':{'cost_usd':0,'calls':0}}
    
    total_in = msg1.usage.input_tokens
    total_out = msg1.usage.output_tokens
    
    header, rows, parse_err = _parse_claude_response(msg1)
    if parse_err:
        cost = (total_in*3 + total_out*15)/1_000_000
        return {'header':header,'data':rows,'mode':'claude_vision','tier':3,
                'error':parse_err,'balance_stats':{},
                'api_usage':{'cost_usd':round(cost,4),'calls':1,'input_tokens':total_in,'output_tokens':total_out}}
    
    # Check balance
    df = validate_balance(pd.DataFrame(rows))
    st = balance_stats(df)
    active_fails = df[(df[DATA_COLUMNS].sum(axis=1)>0) & (~df['balanced'])]
    
    if active_fails.empty:
        cost = (total_in*3 + total_out*15)/1_000_000
        return {'header':header,'data':rows,'mode':'claude_vision',
                'tier':1 if st['pass_rate']>=0.75 else 2,'error':None,'balance_stats':st,
                'api_usage':{'cost_usd':round(cost,4),'calls':1,'input_tokens':total_in,'output_tokens':total_out}}
    
    # ── Pass 2: retry unbalanced rows ──
    fail_info = [{'code':int(r['Code']),'product':PRODUCT_CODES.get(int(r['Code'])),'values':{c:r[c] for c in DATA_COLUMNS},'diff':r['balance_diff']} for _,r in active_fails.iterrows()]
    retry_prompt = f'''These rows failed the balance check (storage_bom + receipts + products_mfg should equal input + fuel_used + deliveries + storage_eom):\n\n{json.dumps(fail_info, indent=2)}\n\nRe-read ONLY these rows from the image, matching numbers to the correct product name.\nReturn ONLY a JSON array:\n[{{"code":1,"storage_bom":0,"receipts":0,"input":0,"products_mfg":0,"fuel_used":0,"deliveries":0,"storage_eom":0}}]'''
    
    msg2, err2 = _call_claude(b64, retry_prompt, model)
    if err2:
        cost = (total_in*3 + total_out*15)/1_000_000
        return {'header':header,'data':rows,'mode':'claude_vision',
                'tier':1 if st['pass_rate']>=0.75 else 2,'error':None,'balance_stats':st,
                'api_usage':{'cost_usd':round(cost,4),'calls':1,'retry_error':err2,'input_tokens':total_in,'output_tokens':total_out}}
    
    total_in += msg2.usage.input_tokens
    total_out += msg2.usage.output_tokens
    
    # Parse corrections
    txt2 = re.sub(r'^```(?:json)?\s*','', msg2.content[0].text.strip())
    txt2 = re.sub(r'\s*```$','', txt2)
    try:
        corrections = json.loads(txt2)
        corrected = {int(r['code']):r for r in corrections}
        for row in rows:
            if row['Code'] in corrected:
                c = corrected[row['Code']]
                for k,v in [('storage_bom','Storage Beginning of Month'),('receipts','Receipts'),('input','Input'),('products_mfg','Products Manufactured'),('fuel_used','Fuel Used'),('deliveries','Deliveries'),('storage_eom','Storage End of Month')]:
                    row[v] = float(c.get(k, 0))
    except Exception as _retry_err:
        print(f'    Warning: retry-correction parse failed ({type(_retry_err).__name__}: {_retry_err}); keeping Pass-1 values.')
    
    df2 = validate_balance(pd.DataFrame(rows))
    st2 = balance_stats(df2)
    cost = (total_in*3 + total_out*15)/1_000_000
    
    return {'header':header,'data':rows,'mode':'claude_vision',
            'tier':1 if st2['pass_rate']>=0.75 else 2,'error':None,'balance_stats':st2,
            'api_usage':{'cost_usd':round(cost,4),'calls':2,'retried_rows':len(active_fails),'input_tokens':total_in,'output_tokens':total_out}}

print('Tier 3 loaded.')

Tier 3 loaded.


In [7]:
# ── Unified extractor ──

def extract_r1(path, use_claude=True, claude_threshold=0.7):
    if is_digital_pdf(path):
        r = extract_digital(path)
        if r['data']: return r
    r = extract_scanned(path)
    if r.get('data') and r.get('balance_stats',{}).get('pass_rate',0) >= claude_threshold:
        return r
    t2 = r
    cr = None
    if use_claude and HAS_ANTHROPIC:
        cr = extract_claude_with_retry(path)
        if not cr.get('error'): return cr
    if t2.get('data'):
        t2['note'] = 'Low confidence'
        return t2
    # Propagate Claude's real error and any partial cost so the batch log is informative.
    claude_err = cr.get('error', 'claude_not_called') if cr else 'claude_not_called'
    claude_api = cr.get('api_usage', {'cost_usd': 0, 'calls': 0}) if cr else {'cost_usd': 0, 'calls': 0}
    return {'header': {}, 'data': [], 'mode': 'failed', 'tier': 3,
            'error': f'All failed | Claude: {claude_err}',
            'balance_stats': {}, 'api_usage': claude_api}

print('Unified extractor ready.')

Unified extractor ready.


---
## 2. Crash-Safe Batch Extraction

Saves results after EVERY PDF. If the loop crashes, restart and it picks up where it left off.
No duplicate API calls.

In [8]:
def load_checkpoint():
    data_path = CHECKPOINT_DIR / 'extracted_rows.csv'
    log_path = CHECKPOINT_DIR / 'extraction_log.csv'
    
    try:
        data_df = pd.read_csv(data_path, dtype={'id': str}) if data_path.exists() and data_path.stat().st_size > 0 else pd.DataFrame()
    except Exception:
        data_df = pd.DataFrame()
    
    try:
        log_df = pd.read_csv(log_path, dtype={'id': str}) if log_path.exists() and log_path.stat().st_size > 0 else pd.DataFrame()
    except Exception:
        log_df = pd.DataFrame()
    
    done = set(log_df['file'].tolist()) if not log_df.empty and 'file' in log_df.columns else set()
    return data_df, log_df, done


def _validate_api_key():
    """Fast-fail check before a batch run. Returns (ok, error_message)."""
    if not HAS_ANTHROPIC:
        return False, 'anthropic package not installed'
    key = os.environ.get('ANTHROPIC_API_KEY', '').strip()
    if not key:
        return False, 'ANTHROPIC_API_KEY is not set — run Cell 1 and check your .env file'
    if not key.startswith('sk-ant-'):
        return False, f'ANTHROPIC_API_KEY looks wrong (starts with {key[:8]!r})'
    return True, None


# ── Rotation cache: maps refinery_id → the image rotation (0/90/-90) that worked ──
# Stored in output/checkpoints/rotation_cache.json so it persists across sessions.
ROTATION_CACHE_PATH = CHECKPOINT_DIR / 'rotation_cache.json'

def load_rotation_cache():
    """Return the {refinery_id: rotation} dict from disk, or {} if not found."""
    if ROTATION_CACHE_PATH.exists():
        try:
            return json.loads(ROTATION_CACHE_PATH.read_text())
        except Exception:
            return {}
    return {}

def save_rotation_cache(cache):
    """Write the rotation cache to disk immediately (so a crash doesn't lose it)."""
    ROTATION_CACHE_PATH.write_text(json.dumps(cache, indent=2))


def batch_extract(data_dir=RAW_DATA_DIR, facility_filter=None, use_claude=True,
                  resume=True, delay_between_api=1.0):
    # ── Validate API key upfront so we don't silently fail on every PDF ──
    if use_claude:
        ok, key_err = _validate_api_key()
        if not ok:
            print(f'ERROR: {key_err}')
            print('Set ANTHROPIC_API_KEY or pass use_claude=False to run Tier 1/2 only.')
            return pd.DataFrame(), pd.DataFrame()

    pdfs = sorted(data_dir.rglob('*.pdf'))
    if facility_filter:
        pdfs = [p for p in pdfs if any(f in str(p) for f in facility_filter)]
    
    # Load checkpoint if resuming
    if resume:
        prev_data, prev_log, done = load_checkpoint()
        all_rows = prev_data.to_dict('records') if not prev_data.empty else []
        all_log = prev_log.to_dict('records') if not prev_log.empty else []
        n_skip = sum(1 for p in pdfs if str(p) in done)
        print(f'{len(pdfs)} PDFs total, {n_skip} already done, {len(pdfs)-n_skip} remaining.')
    else:
        all_rows, all_log, done = [], [], set()
        for f in CHECKPOINT_DIR.glob('*.csv'): f.unlink()
        print(f'{len(pdfs)} PDFs. Starting fresh.')
    
    def save_progress():
        pd.DataFrame(all_rows).to_csv(CHECKPOINT_DIR / 'extracted_rows.csv', index=False)
        pd.DataFrame(all_log).to_csv(CHECKPOINT_DIR / 'extraction_log.csv', index=False)
    
    processed, total_cost = 0, 0.0
    
    for i, fp in enumerate(pdfs):
        fp_str = str(fp)
        if resume and fp_str in done:
            continue
        
        # Parse filename
        parts = fp.stem.split('___')
        if len(parts) < 2:
            all_log.append({'file': fp_str, 'id': '', 'date': '', 'mode': 'failed', 'tier': 3,
                            'error': 'bad filename', 'claude_error': '',
                            'active': 0, 'balanced': 0, 'pass_rate': 0, 'cost': 0, 'calls': 0, 'retried_rows': 0})
            continue
        
        fid = str(parts[1].split('_')[0])
        try:
            yr, mo = parts[0].split('_')[:2]; yr, mo = int(yr), int(mo)
            date = datetime(yr, mo+1, 1) - pd.Timedelta(days=1) if mo < 12 else datetime(yr, 12, 31)
            date_str = date.strftime('%Y-%m-%d')
        except:
            date_str = ''
        
        # Extract
        try:
            r = extract_r1(fp_str, use_claude=use_claude)
        except Exception as e:
            print(f'\n{"!"*60}')
            print(f'ERROR on file {i+1}/{len(pdfs)}: {fp.name}')
            print(f'{type(e).__name__}: {e}')
            print(f'Saving progress ({processed} PDFs processed, ${total_cost:.2f} spent)...')
            save_progress()
            print(f'Saved to {CHECKPOINT_DIR}')
            print(f'Fix the issue and re-run batch_extract() to resume.')
            print(f'{"!"*60}')
            raise
        
        # Accumulate results in memory
        api = r.get('api_usage', {})
        for row in r.get('data', []):
            row.update({'id': fid, 'date': date_str,
                        'extraction_mode': r['mode'], 'extraction_tier': r.get('tier', 3)})
            all_rows.append(row)
        
        raw_err = r.get('error') or ''
        # Split "All failed | Claude: <detail>" into separate columns for readability.
        if '| Claude:' in (raw_err or ''):
            top_err, claude_detail = raw_err.split('| Claude:', 1)
        else:
            top_err, claude_detail = raw_err, ''

        all_log.append({
            'file': fp_str, 'id': fid, 'date': date_str, 'mode': r['mode'], 'tier': r.get('tier', 3),
            'error': top_err.strip(), 'claude_error': claude_detail.strip(),
            'active': r.get('balance_stats', {}).get('total_active', 0),
            'balanced': r.get('balance_stats', {}).get('balanced_count', 0),
            'pass_rate': r.get('balance_stats', {}).get('pass_rate', 0),
            'cost': api.get('cost_usd', 0), 'calls': api.get('calls', 0),
            'retried_rows': api.get('retried_rows', 0)})
        
        processed += 1
        total_cost += api.get('cost_usd', 0)
        
        if processed % 25 == 0:
            print(f'  [{processed} done] last={r["mode"]} tier={r.get("tier")} cost=${total_cost:.2f}')
        
        if api.get('calls', 0) > 0 and delay_between_api > 0:
            time.sleep(delay_between_api)
    
    # ── All done: save final results ──
    save_progress()
    
    rdf = pd.DataFrame(all_rows)
    ldf = pd.DataFrame(all_log)
    
    print(f'\n{"="*60}')
    print(f'BATCH COMPLETE: {processed} processed, ${total_cost:.2f} API cost')
    if not ldf.empty:
        print(f'Tiers: {dict(ldf["tier"].value_counts().sort_index())}')
        print(f'Modes: {dict(ldf["mode"].value_counts())}')
        failed = ldf[ldf['mode'] == 'failed']
        if not failed.empty:
            print(f'Failed ({len(failed)}):')
            for _, row in failed.iterrows():
                print(f'  {Path(row["file"]).name} → {row.get("claude_error") or row.get("error")}')
    print(f'Saved to {CHECKPOINT_DIR}')
    
    return rdf, ldf

In [9]:
# ── Retry-only pass: re-run every mode='failed' PDF without touching successes ──

# Rotations to attempt when Claude returns an empty/unreadable response.
# Portrait scans of a landscape form are rotated 90°; we try both directions.
_RETRY_ROTATIONS = [0, 90, -90]

def _safe_csv_write(df, path):
    """Write a CSV, with a helpful message if the file is locked (e.g. open in Excel)."""
    try:
        df.to_csv(path, index=False)
        return True
    except PermissionError:
        print(f'\n  PermissionError: cannot write {path}')
        print('  → Close the file in Excel (or any other program) and re-run this cell.')
        return False


def retry_failed(log_csv, data_csv=None, facility_filter=None,
                 model='claude-sonnet-4-6', delay=2.0):
    """Re-attempt extraction for every row where mode='failed' in log_csv.

    Parameters
    ----------
    log_csv         : path to the extraction log CSV
    data_csv        : path to the extracted-data CSV to patch in-place (optional)
    facility_filter : list of id strings to limit retries, e.g. ['01-0692']
    model           : Claude model name
    delay           : seconds to wait between successful API calls
    """
    ok, key_err = _validate_api_key()
    if not ok:
        print(f'ERROR: {key_err}'); return None, None

    log_df = pd.read_csv(log_csv, dtype={'id': str})
    mask = log_df['mode'] == 'failed'
    if facility_filter:
        mask &= log_df['id'].isin(facility_filter)
    targets = log_df[mask].copy()

    if targets.empty:
        print('No failed rows to retry.'); return log_df, None

    print(f'Retrying {len(targets)} failed PDFs...')

    data_df = pd.read_csv(data_csv, dtype={'id': str}) if data_csv and Path(data_csv).exists() else pd.DataFrame()
    new_rows, updated_log = [], []
    total_cost = 0.0  # cumulative across the whole retry run
    cache = load_rotation_cache()  # {refinery_id: known-good rotation}

    for _, row in targets.iterrows():
        fp = row['file']
        fid = str(row['id'])

        if not Path(fp).exists():
            print(f'  SKIP (not found): {fp}')
            updated_log.append(row.to_dict())
            continue

        print(f'  {Path(fp).name}', end=' ... ', flush=True)

        # Build rotation order: try the cached rotation first if we have one,
        # then fall back to the remaining options so we never miss a file.
        cached_rot = cache.get(fid)
        if cached_rot is not None:
            rotations = [cached_rot] + [r for r in _RETRY_ROTATIONS if r != cached_rot]
            print(f'[cache: {cached_rot:+d}°]', end=' ', flush=True)
        else:
            rotations = _RETRY_ROTATIONS

        # ── Try each rotation until Claude returns parseable JSON ──
        success_result = None   # (header, rows, working_rotation) on success
        file_cost = 0.0         # cost accumulated for this one PDF (across all rotation attempts)
        last_err = 'no attempts made'

        for rotation in rotations:
            # Rasterize + optionally rotate
            try:
                img = rasterize_page(fp, 300)
                if rotation != 0:
                    img = img.rotate(rotation, expand=True)
                    if cached_rot != rotation:  # only print when it's not the cached (expected) rotation
                        print(f'\n    [retrying at {rotation:+d}°]', end=' ', flush=True)
                b64 = image_to_base64(img)
            except Exception as e:
                last_err = f'RasterizeError: {e}'
                print(f'rasterize FAIL: {e}')
                break  # no point rotating if we can't even read the file

            # Call Claude
            msg, api_err = _call_claude(b64, CLAUDE_PROMPT, model)
            if api_err:
                last_err = api_err
                print(f'Claude FAIL: {api_err}')
                break  # API errors are not rotation-dependent

            # Track cost regardless of whether the parse succeeds
            in_tok = msg.usage.input_tokens; out_tok = msg.usage.output_tokens
            attempt_cost = round((in_tok * 3 + out_tok * 15) / 1_000_000, 4)
            file_cost += attempt_cost

            # Inspect raw response before parsing
            raw_text = msg.content[0].text.strip()
            if not raw_text:
                last_err = 'empty response from Claude'
                print(f'[empty, cost=${attempt_cost:.4f}]', end=' ', flush=True)
                continue  # try next rotation

            # Show a snippet whenever the text isn't valid JSON so you can see what Claude said
            header, rows, parse_err = _parse_claude_response(msg)
            if parse_err:
                last_err = f'{parse_err} | Claude said: {repr(raw_text[:120])}'
                print(f'[parse fail, cost=${attempt_cost:.4f}] Claude said: {repr(raw_text[:120])}', flush=True)
                continue  # try next rotation

            # Success — remember which rotation worked for this refinery
            success_result = (header, rows, rotation)
            if cache.get(fid) != rotation:
                cache[fid] = rotation
                save_rotation_cache(cache)
            break

        total_cost += file_cost

        if success_result is None:
            print(f'FAIL (all rotations)  cost=${file_cost:.4f}  [session total ${total_cost:.4f}]')
            r = row.to_dict()
            r['claude_error'] = last_err
            r['cost'] = file_cost
            updated_log.append(r)
            time.sleep(delay)
            continue

        header, rows, working_rotation = success_result
        # Use validate_balance output so balance_diff / balanced come along for free
        df = validate_balance(pd.DataFrame(rows))
        st = balance_stats(df)
        tier = 1 if st['pass_rate'] >= 0.75 else 2 if st['pass_rate'] >= 0.25 else 3

        print(f'OK  pass_rate={st["pass_rate"]:.0%}  cost=${file_cost:.4f}  [session total ${total_cost:.4f}]')

        date_str = row['date']
        for rw in df.to_dict('records'):   # ← df includes balance_diff & balanced
            rw['material'] = PRODUCT_CODES.get(int(rw['Code']), '')
            rw.update({'id': fid, 'date': date_str,
                       'extraction_mode': 'claude_vision', 'extraction_tier': tier})
            new_rows.append(rw)

        r = row.to_dict()
        r.update({'mode': 'claude_vision', 'tier': tier, 'error': '',
                  'claude_error': '', 'active': st['total_active'],
                  'balanced': st['balanced_count'], 'pass_rate': st['pass_rate'],
                  'cost': file_cost, 'calls': 1, 'retried_rows': 0})
        updated_log.append(r)
        time.sleep(delay)

    # ── Merge results back into the original log ──
    result_log = log_df.copy()
    update_map = {r['file']: r for r in updated_log}
    for col in ['claude_error']:
        if col not in result_log.columns:
            result_log[col] = ''
    for idx, row in result_log.iterrows():
        if row['file'] in update_map:
            for k, v in update_map[row['file']].items():
                if k in result_log.columns:
                    result_log.at[idx, k] = v

    # ── Patch data CSV ──
    if data_csv and new_rows:
        new_df = pd.DataFrame(new_rows)
        if not data_df.empty:
            keep = ~((data_df['id'].isin(targets['id'])) &
                     (data_df['date'].isin(targets['date'])))
            data_df = pd.concat([data_df[keep], new_df], ignore_index=True)
        else:
            data_df = new_df
        if _safe_csv_write(data_df, data_csv):
            print(f'Data saved → {data_csv}')

    if _safe_csv_write(result_log, log_csv):
        print(f'Log  saved → {log_csv}')

    n_ok = sum(1 for r in updated_log if r.get('mode') == 'claude_vision')
    n_fail = len(targets) - n_ok
    print(f'\n{"="*55}')
    print(f'Retry complete: {n_ok} recovered, {n_fail} still failed')
    print(f'Session API cost: ${total_cost:.4f}')
    print(f'{"="*55}')
    return result_log, data_df

print('retry_failed() ready.')

retry_failed() ready.


In [ ]:
# ── Run retry for 01-0692 failed PDFs ──
# • Tries original orientation, then +90° and -90° rotation if Claude returns empty text.
# • Prints Claude's raw reply snippet when parse fails so you can diagnose the issue.
# • Tracks cost per file AND as a running session total.
# • Close any open Excel files before running — they lock the CSVs.

retry_log, retry_data = retry_failed(
    log_csv  = OUTPUT_DIR / 'extraction_log_01-0717.csv',
    data_csv = OUTPUT_DIR / 'extracted_data_01-0717.csv',
    facility_filter = ['01-0717'],
)

# Inspect which ones still fail and why:
if retry_log is not None:
    still_failed = retry_log[(retry_log['mode'] == 'failed') & (retry_log['id'] == '01-0692')]
    if still_failed.empty:
        print('All previously-failed 01-0692 PDFs now have data.')
    else:
        print(f'\n{len(still_failed)} still failing — Claude raw error:')
        cols = [c for c in ['file', 'error', 'claude_error', 'cost'] if c in still_failed.columns]
        print(still_failed[cols].to_string(index=False))

Retrying 9 failed PDFs...
  2020_10___01-0692_THE SAN ANTONIO REFINERY LLC.pdf ... [parse fail, cost=$0.0245] Claude said: 'I\'ll carefully extract each row from the form, reading the columns systematically.\n\n{"operator":"The San Antonio Refiner'

    [retrying at +90°] [parse fail, cost=$0.0365] Claude said: 'I need to carefully read this upside-down form. Let me analyze each row systematically.\n\nThe form is rotated 180°, so I '

    [retrying at -90°] OK  pass_rate=100%  cost=$0.0851  [session total $0.0851]
  2020_11___01-0692_THE SAN ANTONIO REFINERY LLC.pdf ... [parse fail, cost=$0.0244] Claude said: 'I\'ll carefully read each row from the document.\n\n{"operator":"The San Antonio Refinery, LLC","month":"November 2020","re'

    [retrying at +90°] [parse fail, cost=$0.0336] Claude said: 'I need to carefully read this upside-down form. Let me rotate mentally and extract each row.\n\nThe form is upside down, s'

    [retrying at -90°] OK  pass_rate=100%  cost=$0.0821  [session t

In [ ]:
# ── Run batch extraction ──

# Full run (resumes from checkpoint if interrupted):
# new_data, new_log = batch_extract()

# Specific facilities:
# new_data, new_log = batch_extract(facility_filter=['03-0052','03-0138','02-0057'])

facility = ['01-0717']
new_data, new_log = batch_extract(facility_filter=facility, use_claude=True)
# Without Claude API (Tesseract only, free):
# new_data, new_log = batch_extract(use_claude=False)

# Force reprocess everything (ignore checkpoint):
# new_data, new_log = batch_extract(resume=False)

# Copy final results from checkpoint to output:
#new_data.to_csv(OUTPUT_DIR / 'extracted_data.csv', index=False)
#new_log.to_csv(OUTPUT_DIR / 'extraction_log.csv', index=False)


# Add balance check columns
if not new_data.empty:
    new_data['balance_diff'] = (
        new_data['Storage Beginning of Month'] + new_data['Receipts'] + new_data['Products Manufactured']
    ) - (
        new_data['Input'] + new_data['Fuel Used'] + new_data['Deliveries'] + new_data['Storage End of Month']
    )
    new_data['balanced'] = new_data['balance_diff'].abs() < 2
    new_data['material'] = new_data['Code'].map(PRODUCT_CODES)
    
    # Summary
    active = new_data[new_data[DATA_COLUMNS].sum(axis=1) > 0]
    print(f'Active rows: {len(active)}, Balanced: {active["balanced"].sum()} ({active["balanced"].mean():.0%})')
    print(f'Rows needing review: {(~active["balanced"]).sum()}')

new_data.to_csv(OUTPUT_DIR / f'extracted_data_{facility[0]}.csv', index=False)
new_log.to_csv(OUTPUT_DIR / f'extraction_log_{facility[0]}.csv', index=False)
new_data[new_data['balanced'] == False].groupby(['id','date']).size().reset_index().to_csv(OUTPUT_DIR / f'reports_to_review_count.csv', index=False)


if "api_calls" in new_log.columns: print(f'PDFs needing retry: {(new_log["api_calls"] == 2).sum()}') 
print(f'Total retried rows: {new_log["retried_rows"].sum()}')

#print('Batch cell ready.')

---
## 3. Quick Test: Single PDFs

In [ ]:
def test_pdf(path, use_claude=True):
    if not os.path.exists(path): print(f'Not found: {path}'); return None
    r = extract_r1(path, use_claude=use_claude)
    print(f'File: {os.path.basename(path)}')
    print(f'Mode: {r["mode"]} | Tier: {r.get("tier")} | Header: {r["header"]}')
    if r.get('error'): print(f'Error: {r["error"]}')
    s = r.get('balance_stats',{})
    if s: print(f'Balance: {s.get("balanced_count",0)}/{s.get("total_active",0)} ({s.get("pass_rate",0):.0%})')
    a = r.get('api_usage',{})
    if a.get('calls',0)>0: print(f'API: {a["calls"]} call(s), ${a["cost_usd"]:.4f}')
    if not r['data']: return None
    df = validate_balance(pd.DataFrame(r['data']))
    df['material'] = df['Code'].map(PRODUCT_CODES)
    df['confidence'] = df['balanced'].map({True:'ok',False:'REVIEW'})
    pd.set_option('display.float_format','{:,.0f}'.format)
    print(df[['Code','material']+DATA_COLUMNS+['balance_diff','confidence']].to_string(index=False))
    return df

In [ ]:
# df = test_pdf('2025-june-03-0052.pdf')         # Digital (Blanchard)
# df = test_pdf('2026-january-03-0138.pdf')       # Clean scan (ExxonMobil)
# df = test_pdf('2025_1___02-0057_DIAMOND SHAMROCK REFINING COMPANY.pdf')  # Difficult scan
print('Test cell ready.')

---
## 4. Compare to Existing Database

In [ ]:
def compare(new_df, existing_path=EXISTING_DB_PATH):
    ex = pd.read_excel(existing_path, dtype={'id':str})
    ex['date'] = pd.to_datetime(ex['date'])
    nw = new_df.copy(); nw['date'] = pd.to_datetime(nw['date'])
    m = nw.merge(ex, on=['id','date','Code'], how='outer', suffixes=('_new','_old'), indicator=True)
    new_only = m[m['_merge']=='left_only']
    both = m[m['_merge']=='both']
    oc = [f'{c}_old' for c in DATA_COLUMNS if f'{c}_old' in both.columns]
    nc = [f'{c}_new' for c in DATA_COLUMNS if f'{c}_new' in both.columns]
    silent = both[(both[oc].fillna(0).sum(axis=1)==0)&(both[nc].fillna(0).sum(axis=1)>0)]
    print(f'Both: {len(both)} | Silent zeros: {len(silent)} | New only: {len(new_only)}')
    return {'silent_zeros':silent,'new_only':new_only}

# comp = compare(new_data)
print('Compare cell ready.')

---
## 5. Merge & Save

In [ ]:
def merge_db(new_df, existing_path=EXISTING_DB_PATH):
    ex = pd.read_excel(existing_path, dtype={'id':str})
    ex['date'] = pd.to_datetime(ex['date'])
    nw = new_df[['id','date','Code']+DATA_COLUMNS].copy()
    nw['date'] = pd.to_datetime(nw['date'])
    mg = ex.merge(nw, on=['id','date','Code'], how='outer', suffixes=('','_new'))
    for col in DATA_COLUMNS:
        nc = f'{col}_new'
        if nc in mg.columns:
            z=mg[col].fillna(0)==0; h=mg[nc].fillna(0)!=0
            mg.loc[z&h,col]=mg.loc[z&h,nc]
            mg[col]=mg[col].fillna(mg[nc])
            mg.drop(columns=[nc],inplace=True)
    return mg.sort_values(['id','date','Code']).reset_index(drop=True)

# final = merge_db(new_data)
# final.to_csv(OUTPUT_DIR / 'rrc_r1_database.csv', index=False)
print('Merge cell ready.')

---
## Notes

**Crash recovery:** Results save to `output/checkpoints/` after every PDF. If the loop fails:
1. Fix the issue (API key, network, etc.)
2. Re-run `batch_extract()` — it skips already-processed PDFs automatically
3. No duplicate API calls, no lost data

To start completely fresh: `batch_extract(resume=False)`

**Image size limit:** Images auto-compress to JPEG under 4.5MB before sending to Claude.

**Cost tracking:** Every result includes `api_usage.cost_usd` (0 for Tier 1/2). Batch summary shows total.

| Tier | Method | Accuracy | Cost |
|------|--------|----------|------|
| 1 | pdfplumber | ~100% | Free |
| 2 | Tesseract | 60-75% | Free |
| 3 | Claude vision + retry | ~95%+ | ~$0.01-0.05/page |